# 🦽 AccessMap ML — Training on Google Colab

Train a YOLOv8s model for accessibility feature detection using Google Colab's free GPU.

**Requirements:**
- Google Colab with GPU runtime (T4 recommended)
- Your labeled dataset uploaded to Google Drive

**Classes (10):**
wheelchair, wheelchair_ramp, elevator, lift, stairs, accessible_parking, accessible_bathroom, grab_bar, doorway, tactile_paving

## 1. Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

# Install dependencies
!pip install ultralytics>=8.3.0 albumentations>=1.4.0 loguru pyyaml -q

# Verify installation
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 2. Mount Google Drive & Upload Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set your dataset path in Google Drive
# Expected structure:
#   MyDrive/accessmap-data/
#     images/
#       train/
#       val/
#       test/
#     labels/
#       train/
#       val/
#       test/

DRIVE_DATA_PATH = '/content/drive/MyDrive/accessmap-data'
DRIVE_OUTPUT_PATH = '/content/drive/MyDrive/accessmap-models'

import os
os.makedirs(DRIVE_OUTPUT_PATH, exist_ok=True)
print(f'Data path: {DRIVE_DATA_PATH}')
print(f'Output path: {DRIVE_OUTPUT_PATH}')

## 3. Create Dataset Configuration

In [ ]:
import yaml

# Create dataset.yaml for YOLOv8
dataset_config = {
    'path': DRIVE_DATA_PATH,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 10,
    'names': {
        0: 'wheelchair',
        1: 'wheelchair_ramp',
        2: 'elevator',
        3: 'lift',
        4: 'stairs',
        5: 'accessible_parking',
        6: 'accessible_bathroom',
        7: 'grab_bar',
        8: 'doorway',
        9: 'tactile_paving',
    }
}

dataset_yaml_path = '/content/dataset.yaml'
with open(dataset_yaml_path, 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False, sort_keys=False)

print('dataset.yaml created!')
print(yaml.dump(dataset_config, default_flow_style=False))

## 4. Verify Dataset

In [ ]:
from pathlib import Path

data_path = Path(DRIVE_DATA_PATH)

for split in ['train', 'val', 'test']:
    img_dir = data_path / 'images' / split
    lbl_dir = data_path / 'labels' / split
    
    if img_dir.exists():
        n_images = len(list(img_dir.glob('*.jpg'))) + len(list(img_dir.glob('*.png')))
        n_labels = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
        print(f'{split}: {n_images} images, {n_labels} labels')
    else:
        print(f'{split}: directory not found at {img_dir}')

## 5. Train the Model 🚀

In [ ]:
from ultralytics import YOLO

# Load pre-trained YOLOv8s model
model = YOLO('yolov8s.pt')

# Train!
results = model.train(
    data=dataset_yaml_path,
    epochs=150,
    imgsz=640,
    batch=16,                  # Reduce to 8 if you get OOM errors
    patience=30,               # Early stopping
    optimizer='AdamW',
    lr0=0.001,
    weight_decay=0.0005,
    warmup_epochs=5,
    cos_lr=True,
    freeze=10,                 # Freeze backbone for small datasets
    
    # Augmentation
    fliplr=0.5,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    
    # Output
    project='/content/runs',
    name='accessmap_train',
    exist_ok=True,
    plots=True,
    verbose=True,
    seed=42,
)

## 6. Evaluate on Test Set

In [ ]:
# Load best model
best_model = YOLO('/content/runs/accessmap_train/weights/best.pt')

# Evaluate on test set
test_results = best_model.val(
    data=dataset_yaml_path,
    split='test',
    imgsz=640,
    plots=True,
    verbose=True,
)

# Print results
print(f'\nmAP@0.5:      {test_results.box.map50:.4f}')
print(f'mAP@0.5:0.95: {test_results.box.map:.4f}')
print(f'Precision:     {test_results.box.mp:.4f}')
print(f'Recall:        {test_results.box.mr:.4f}')

## 7. View Training Results

In [ ]:
from IPython.display import Image, display

# Display training curves
results_dir = Path('/content/runs/accessmap_train')

for plot_name in ['results.png', 'confusion_matrix_normalized.png', 'PR_curve.png', 'F1_curve.png']:
    plot_path = results_dir / plot_name
    if plot_path.exists():
        print(f'\n{plot_name}:')
        display(Image(filename=str(plot_path), width=800))

## 8. Test Inference on Sample Images

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Get test images
test_img_dir = Path(DRIVE_DATA_PATH) / 'images' / 'test'
test_images = sorted(list(test_img_dir.glob('*.jpg')) + list(test_img_dir.glob('*.png')))[:6]

if test_images:
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    for i, img_path in enumerate(test_images):
        ax = axes[i // 3][i % 3]
        
        results = best_model(str(img_path), conf=0.25, verbose=False)
        annotated = results[0].plot()
        
        ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        ax.set_title(img_path.name, fontsize=10)
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('/content/sample_predictions.png', dpi=150)
    plt.show()
else:
    print('No test images found')

## 9. Export Model

In [ ]:
# Export to ONNX
onnx_path = best_model.export(format='onnx', imgsz=640, opset=17, simplify=True)
print(f'ONNX model exported to: {onnx_path}')

# Export to TFLite (for mobile)
try:
    tflite_path = best_model.export(format='tflite', imgsz=640)
    print(f'TFLite model exported to: {tflite_path}')
except Exception as e:
    print(f'TFLite export failed: {e}')

# Show model sizes
import os
pt_size = os.path.getsize('/content/runs/accessmap_train/weights/best.pt') / 1e6
onnx_size = os.path.getsize(onnx_path) / 1e6
print(f'\nModel sizes:')
print(f'  PyTorch: {pt_size:.1f} MB')
print(f'  ONNX:    {onnx_size:.1f} MB')

## 10. Save to Google Drive

In [ ]:
import shutil

# Copy models to Google Drive
output_dir = Path(DRIVE_OUTPUT_PATH)

# Copy best weights
shutil.copy('/content/runs/accessmap_train/weights/best.pt', output_dir / 'best.pt')
shutil.copy('/content/runs/accessmap_train/weights/last.pt', output_dir / 'last.pt')

# Copy ONNX
if os.path.exists(onnx_path):
    shutil.copy(onnx_path, output_dir / 'best.onnx')

# Copy training plots
plots_dir = output_dir / 'plots'
plots_dir.mkdir(exist_ok=True)
for plot in ['results.png', 'confusion_matrix_normalized.png', 'PR_curve.png', 'F1_curve.png']:
    src = Path('/content/runs/accessmap_train') / plot
    if src.exists():
        shutil.copy(src, plots_dir / plot)

print(f'\n✅ Models and plots saved to Google Drive: {DRIVE_OUTPUT_PATH}')
print('Files saved:')
for f in sorted(output_dir.rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to(output_dir)} ({f.stat().st_size / 1e6:.1f} MB)')

## ✅ Done!

Your trained model is saved to Google Drive. Next steps:

1. Download `best.pt` to your local `access-map-ml/models/` directory
2. Start the API server: `uvicorn src.api.main:app --host 0.0.0.0 --port 8000`
3. Test: `curl -X POST http://localhost:8000/detect -F 'file=@test.jpg'`